# 3_Merge

**Objective:**
This notebook enriches the Seasoned Equity Offering (SEO) dataset with a comprehensive set of macroeconomic, industry, and market-level indicators. The aim is to provide a broader economic and financial context for each SEO, enabling more accurate modeling and interpretation of underpricing behavior.

**Key Tasks Completed:**
- State-Level Economic Indicators:
Integrated Annual Personal Income & Employment (APIE) and Personal Consumption Expenditure (PCE) datasets

Computed lagged year-over-year percent changes in employment and spending metrics

Merged data by issuer state and offer year

- Quarterly and National-Level Indicators:
Merged Quarterly Personal Income (QPI) and Quarterly GDP (QGDP) with lagged quarter-over-quarter growth rates

Integrated Fama-French momentum and 5-factor models, lagged one month

Joined industry portfolio returns (17-industry value-weighted returns) matched by issuer industry and offer month/year

Included lagged U.S. Treasury Bill yields (3-month and 1-year)

- Market Sentiment Indicators:
Merged S&P 500 returns at monthly, weekly, and daily frequencies, lagged appropriately

Integrated VIX Index levels and daily percent changes as a proxy for market volatility

In [1]:
# Install important libraries
#!pip install alpha_vantage
#!pip install pandas_datareader

In [2]:
# ===========================================
# 1. Install and Import Required Libraries
# ===========================================

# !pip install alpha_vantage
# !pip install yfinance
# !pip install pandas_datareader

import pandas as pd
import re
import pyarrow as pa
import warnings
from alpha_vantage.timeseries import TimeSeries
import pandas_datareader.data as web

warnings.filterwarnings('ignore')


In [3]:
# ===========================================
# 2. Define File Paths for Inputs
# ===========================================

seo_state_path = r'data\Updated SEO ISSUES Dataset\SEO_ISSUES_After_ratios_and_pe_state.parquet'
apie_path = r'data\Macroeconomic data\Formatted\2008\Annual Personal Income and Employment.parquet'
pce_path = r'data\Macroeconomic data\Formatted\2008\Personal consumption expenditures.parquet'
qpi_path = r'data\Macroeconomic data\Formatted\2008\Quarterly personal income.parquet'
qgdp_path = r'data\Macroeconomic data\Formatted\2008\Quarterly GDP.parquet'
dtb3_path = r'data\Daily Treasury Bill\DTB3.csv'
dtb1_path = r'data\Daily Treasury Bill\DTB1YR.csv'


In [4]:
# ===========================================
# 3. Load and Preprocess SEO Dataset
# ===========================================

# Load the SEO dataset with state column already integrated from SEC API
seo_state = pd.read_parquet(seo_state_path)

# Dictionary to map full state names to abbreviations
state_abbrev = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY', 'United States': 'US'
}

valid_abbrevs = set(state_abbrev.values())

# Ensure all states in the dataset are valid abbreviations
seo_state['state'] = seo_state['state'].apply(lambda x: x if x in valid_abbrevs else 'US')

# Fill missing date differences with 0
seo_state['Price_Date_Diff'] = seo_state['Price_Date_Diff'].fillna(0)

# Compute actual offer date by subtracting Price_Date_Diff from Issue_Date
def cal_offer_date(row):
    return row['Issue_Date'] - pd.to_timedelta(row['Price_Date_Diff'], unit='D')

seo_state['Issue_Date'] = pd.to_datetime(seo_state['Issue_Date'])
seo_state['Offer_date'] = seo_state.apply(cal_offer_date, axis=1)

# Derive offer quarter, month, and year
seo_state['Offer_Quarter'] = seo_state['Offer_date'].dt.to_period('Q').astype(str)
seo_state['Offer_Month'] = seo_state['Offer_date'].dt.month
seo_state['Offer_Year'] = seo_state['Offer_Year'].astype(str).str[-4:].astype(int)


In [5]:
# ===========================================
# 4. Load and Process APIE Data
# ===========================================

apie = pd.read_parquet(apie_path)
apie.rename(columns={'GeoName': 'State'}, inplace=True)

# Map state to abbreviation
apie['State_Abbrev'] = apie['State'].map(state_abbrev)
apie['Year'] = pd.to_datetime(apie['Year']).dt.year

# Identify numeric columns for percent change computation
numeric_cols = apie.select_dtypes(include='number').columns.drop('Year', errors='ignore')

# Compute year-over-year percent change grouped by state
apie_pct_change = apie.copy()
apie_pct_change[numeric_cols] = apie.groupby('State')[numeric_cols].pct_change() * 100

# Rename columns to indicate percent change
apie_pct_change.rename(columns={col: f"{col} PCT change" for col in numeric_cols}, inplace=True)

# Lag percent change values by 1 year (previous year context)
cols_to_lag = apie_pct_change.columns.difference(['Year', 'State'])
apie_pct_change[cols_to_lag] = apie_pct_change.groupby('State')[cols_to_lag].shift(1)

# Merge lagged APIE percent change data with SEO dataset on state and year
merged_df = seo_state.merge(
    apie_pct_change,
    how='left',
    left_on=['Offer_Year', 'state'],
    right_on=['Year', 'State_Abbrev']
)

# Drop unused columns after merge
merged_df.drop(columns=['State', 'State_Abbrev', 'Year'], inplace=True)


In [6]:
# ===========================================
# 5. Load and Process PCE Data
# ===========================================

pce = pd.read_parquet(pce_path)

# Keep only relevant expenditure columns
cols_to_keep = ['GeoName', 'Year', 'Goods', 'Services']
pce = pce[[col for col in cols_to_keep if col in pce.columns]]

# Rename columns for clarity
pce.rename(columns={
    'GeoName': 'State',
    'Goods': 'Personal_Expenditure_Goods',
    'Services': 'Personal_Expenditure_Services'
}, inplace=True)

# Add state abbreviations
pce['State_Abbrev'] = pce['State'].map(state_abbrev)
pce['Year'] = pd.to_datetime(pce['Year']).dt.year

# Calculate YoY percent change for numeric values
numeric_cols = pce.select_dtypes(include='number').columns.drop('Year', errors='ignore')
pce_pct_change = pce.copy()
pce_pct_change[numeric_cols] = pce.groupby('State')[numeric_cols].pct_change() * 100
pce_pct_change.rename(columns={col: f"{col} PCT change" for col in numeric_cols}, inplace=True)


In [7]:
# ===========================================
# 6. Lagged Percent Change - PCE
# ===========================================

# Step 1: Lag percent change columns for PCE (exclude non-numeric)
cols_to_lag = pce_pct_change.columns.difference(['Year', 'State'])

# Step 2: Apply lag within each state
pce_pct_change[cols_to_lag] = pce_pct_change.groupby('State')[cols_to_lag].shift(1)

# Step 3: Merge lagged PCE data with main dataset by state and offer year
merged_df = merged_df.merge(
    pce_pct_change,
    how='left',
    left_on=['Offer_Year', 'state'],
    right_on=['Year', 'State_Abbrev']
)

# Drop temporary columns from merge
merged_df.drop(columns=['State', 'State_Abbrev', 'Year'], inplace=True)


In [8]:
# ===========================================
# 7. Lagged Quarterly Change - QPI (Quarterly Personal Income)
# ===========================================

# Load QPI data and prepare for merge
qpi = pd.read_parquet(qpi_path)
qpi.rename(columns={'GeoName': 'State'}, inplace=True)
qpi['State_Abbrev'] = qpi['State'].map(state_abbrev)

# Convert Year to datetime and extract quarter string (e.g., "2009Q2")
qpi['Year'] = pd.to_datetime(qpi['Year'])
qpi['Quarter'] = qpi['Year'].dt.to_period('Q').astype(str)

# Identify numeric columns for change computation
numeric_cols_qtr = qpi.select_dtypes(include='number').columns.drop('Year', errors='ignore')

# Sort and calculate quarter-over-quarter percent change
qpi_qoq_change = qpi.copy()
qpi_qoq_change.sort_values(['State', 'Quarter'], inplace=True)
qpi_qoq_change[numeric_cols_qtr] = qpi_qoq_change.groupby('State')[numeric_cols_qtr].pct_change() * 100

# Rename columns to indicate QoQ change
rename_qoq = {col: f"{col} QoQ change" for col in numeric_cols_qtr}
qpi_qoq_change.rename(columns=rename_qoq, inplace=True)

# Lag QoQ metrics within each state
cols_to_lag = qpi_qoq_change.columns.difference(['Year', 'State', 'Quarter', 'State_Abbrev'])
qpi_qoq_change[cols_to_lag] = qpi_qoq_change.groupby('State')[cols_to_lag].shift(1)

# Merge QPI data with SEO dataset based on quarter and state
merged_df = merged_df.merge(
    qpi_qoq_change,
    how='left',
    left_on=['Offer_Quarter', 'state'],
    right_on=['Quarter', 'State_Abbrev']
)

# Drop merge artifacts
merged_df.drop(columns=['State', 'State_Abbrev', 'Year', 'Quarter'], inplace=True)


In [9]:
# ===========================================
# 8. Lagged Quarterly Change - QGDP (Quarterly GDP)
# ===========================================

# Load and process QGDP data
qgdp = pd.read_parquet(qgdp_path)
qgdp.rename(columns={'GeoName': 'State'}, inplace=True)
qgdp['State_Abbrev'] = qgdp['State'].map(state_abbrev)

# Convert Year and derive Quarter
qgdp['Year'] = pd.to_datetime(qgdp['Year'])
qgdp['Quarter'] = qgdp['Year'].dt.to_period('Q').astype(str)

# Identify numeric columns and compute QoQ change
numeric_cols_qtr = qgdp.select_dtypes(include='number').columns.drop('Year', errors='ignore')
qgdp_qoq_change = qgdp.copy()
qgdp_qoq_change.sort_values(['State', 'Quarter'], inplace=True)
qgdp_qoq_change[numeric_cols_qtr] = qgdp_qoq_change.groupby('State')[numeric_cols_qtr].pct_change() * 100

# Rename columns
rename_qoq = {col: f"{col} QoQ change" for col in numeric_cols_qtr}
qgdp_qoq_change.rename(columns=rename_qoq, inplace=True)

# Lag within each state
cols_to_lag = qgdp_qoq_change.columns.difference(['Year', 'State', 'Quarter', 'State_Abbrev'])
qgdp_qoq_change[cols_to_lag] = qgdp_qoq_change.groupby('State')[cols_to_lag].shift(1)

# Merge lagged GDP data into SEO dataset
merged_df = merged_df.merge(
    qgdp_qoq_change,
    how='left',
    left_on=['Offer_Quarter', 'state'],
    right_on=['Quarter', 'State_Abbrev']
)

# Clean up columns
merged_df.drop(columns=['State', 'State_Abbrev', 'Year', 'Quarter'], inplace=True)


In [10]:
# ===========================================
# 9. Final Touch and Export
# ===========================================

# Create a unified merge date string (for future alignment if needed)
merged_df['Merge Date'] = merged_df['Issue_Date'].dt.strftime("01/%m/%Y")

# Export the final enriched dataset before adding market momentum factors
output_path = r'data\Updated SEO ISSUES Dataset'
merged_df.to_parquet(rf'{output_path}\SEO_ISSUES_MERGED_BEFORE_MOMENTUM_FACTORS.parquet')


Part 2

In [11]:
# ===========================================
# 1. Load Momentum and Fama-French Factors
# ===========================================

# Load monthly momentum factor data
mome_df = pd.read_csv('data/Kenneth Datasets/F-F_Momentum_Factor.CSV', skiprows=13)

# Load monthly Fama-French 5-factor data
factors_df = pd.read_csv('data/Kenneth Datasets/F-F_Research_Data_5_Factors_2x3.csv', skiprows=3)

# Preview momentum data
mome_df.head()


,Unnamed: 0,Mom
0,192701,0.36
1,192702,-2.14
2,192703,3.61
3,192704,4.30
4,192705,3.00


In [12]:
# ===========================================
# 2. Format SEO 'Issue_Date' for Monthly Alignment
# ===========================================

# Convert SEO issue date into YYYYMM format for joining with monthly factor data
merged_df['YearMonth'] = merged_df['Issue_Date'].dt.strftime('%Y%m').astype(int)


In [13]:
# ===========================================
# 3. Clean and Lag Momentum Factor Data
# ===========================================

# Rename date column
mome_df.rename(columns={mome_df.columns[0]: 'YearMonth'}, inplace=True)

# Keep only numeric rows and convert types
mome_df = mome_df[mome_df['YearMonth'].astype(str).str.isnumeric()]
mome_df['YearMonth'] = mome_df['YearMonth'].astype(int)
mome_df['Mom   '] = mome_df['Mom   '].astype(float)

# Lag momentum value by one month
mome_df['Mom   '] = mome_df['Mom   '].shift(1)


In [14]:
# ===========================================
# 4. Clean and Lag Fama-French Factors
# ===========================================

factors_df.rename(columns={factors_df.columns[0]: 'YearMonth'}, inplace=True)
factors_df = factors_df[factors_df['YearMonth'].astype(str).str.isnumeric()]
factors_df['YearMonth'] = factors_df['YearMonth'].astype(int)

# Convert all factor columns to float
for col in ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']:
    factors_df[col] = factors_df[col].astype(float)

# Lag each factor by one month
factors_df[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']] = factors_df[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']].shift(1)


In [15]:
# ===========================================
# 5. Merge Momentum and Factor Data into SEO Dataset
# ===========================================

# First merge: momentum
merged_df = pd.merge(merged_df, mome_df, on='YearMonth', how='left')

# Second merge: Fama-French 5 factors
merged_df = pd.merge(merged_df, factors_df, on='YearMonth', how='left')

# Optional preview
merged_df[['Issue_Date', 'YearMonth', 'Mom   ', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']].head()


,Issue_Date,YearMonth,Mom,Mkt-RF,SMB,HML,RMW,CMA,RF
0,2010-01-01,201001,3.01,2.75,6.24,-0.16,1.02,-0.09,0.01
1,2010-01-03,201001,3.01,2.75,6.24,-0.16,1.02,-0.09,0.01
2,2010-01-04,201001,3.01,2.75,6.24,-0.16,1.02,-0.09,0.01
3,2010-01-04,201001,3.01,2.75,6.24,-0.16,1.02,-0.09,0.01
4,2010-01-05,201001,3.01,2.75,6.24,-0.16,1.02,-0.09,0.01


In [16]:
# ===========================================
# 6. Load and Prepare Industry Momentum (17-Portfolios)
# ===========================================

# Load value-weighted monthly returns
vw_data = pd.read_parquet('data/Portfolio 17/Formatted/value_weighted_monthly.parquet')

# Load and clean industry-category mapping
cat17_mapping = pd.read_excel('data/Portfolio 17/Industries_Cat17 mappings.xlsx')
cat17_cleaned = cat17_mapping.iloc[2:].dropna().copy()
cat17_cleaned.columns = ['Industry', 'cat17']
cat17_cleaned['cat17'] = cat17_cleaned['cat17'].str.strip().str.replace(')', '', regex=False)


In [17]:
# ===========================================
# 7. Melt VW Data and Merge with Category Mapping
# ===========================================

# Parse year and month from index column (e.g., 192607)
vw_data['Year'] = vw_data['Unnamed: 0'] // 100
vw_data['Month'] = vw_data['Unnamed: 0'] % 100
vw_data.drop(columns='Unnamed: 0', inplace=True)

# Reshape to long format for merging
vw_data_melted = vw_data.melt(id_vars=['Year', 'Month'], var_name='cat17', value_name='Return')
vw_data_melted['cat17'] = vw_data_melted['cat17'].str.strip()

# Merge to attach industry name
cat17industries = vw_data_melted.merge(cat17_cleaned, on='cat17', how='left')

# Clean column names
cat17industries.rename(columns={"Return": "Industry Monthly Return"}, inplace=True)


In [18]:
# ===========================================
# 8. Lag Industry Return by Industry and Merge
# ===========================================

# Lag industry monthly return by 1 month
cols_to_lag = cat17industries.columns.difference(['Year', 'Industry'])
cat17industries[cols_to_lag] = cat17industries.groupby('Industry')[cols_to_lag].shift(1)

# Merge into SEO data using Offer_Year, Offer_Month, and Issuer_SDC_Industry
merged_df = merged_df.merge(
    cat17industries,
    how='left',
    left_on=['Offer_Year', 'Offer_Month', 'Issuer_SDC_Industry'],
    right_on=['Year', 'Month', 'Industry']
)

# Drop helper columns after merge
merged_df = merged_df.drop(columns=['Industry', 'Date'], errors='ignore')


In [19]:
# ===========================================
# 9. Merge Lagged Treasury Bill Rates
# ===========================================

# Load and prepare 3-month T-bill data
df_DTB3 = pd.read_csv(dtb3_path)
df_DTB3['observation_date'] = pd.to_datetime(df_DTB3['observation_date'])
df_DTB3.set_index('observation_date', inplace=True)
df_DTB3 = df_DTB3.reindex(pd.date_range(df_DTB3.index.min(), df_DTB3.index.max(), freq='D'))
df_DTB3.fillna(method='ffill', inplace=True)
df_DTB3['Last_day_T_Bill_3M_Rate'] = df_DTB3['DTB3'].shift(1)
df_DTB3.reset_index(inplace=True)
df_DTB3.rename(columns={'index': 'Date'}, inplace=True)
df_DTB3.drop(columns='DTB3', inplace=True)

# Load and prepare 1-year T-bill data
df_DTB1YR = pd.read_csv(dtb1_path)
df_DTB1YR['observation_date'] = pd.to_datetime(df_DTB1YR['observation_date'])
df_DTB1YR.set_index('observation_date', inplace=True)
df_DTB1YR = df_DTB1YR.reindex(pd.date_range(df_DTB1YR.index.min(), df_DTB1YR.index.max(), freq='D'))
df_DTB1YR.fillna(method='ffill', inplace=True)
df_DTB1YR['Last_day_T_Bill_1YR_Rate'] = df_DTB1YR['DTB1YR'].shift(1)
df_DTB1YR.reset_index(inplace=True)
df_DTB1YR.rename(columns={'index': 'Date'}, inplace=True)
df_DTB1YR.drop(columns='DTB1YR', inplace=True)


In [20]:
# ===========================================
# 10. Merge Treasury Bill Rates by Offer Date
# ===========================================

# Merge 1-year T-bill
merged_df = merged_df.merge(
    df_DTB1YR,
    how='left',
    left_on='Offer_date',
    right_on='Date'
)
merged_df.drop(columns='Date', inplace=True)

# Merge 3-month T-bill
merged_df = merged_df.merge(
    df_DTB3,
    how='left',
    left_on='Offer_date',
    right_on='Date'
)
merged_df.drop(columns='Date', inplace=True)


In [21]:
# ===========================================
# 11. Fetch and Integrate S&P 500 Daily Data
# ===========================================
from alpha_vantage.timeseries import TimeSeries

API_KEY = "3H84BAQGPVLU9G02"

# Fetch full historical daily data for SPY (S&P 500 ETF)
ts = TimeSeries(key=API_KEY, output_format="pandas")
sp500_data, meta_data = ts.get_daily(symbol="SPY", outputsize="full")

# Convert index to datetime and filter relevant range
sp500_data.index = pd.to_datetime(sp500_data.index)
sp500_data = sp500_data[(sp500_data.index >= "2001-01-01") & (sp500_data.index <= "2009-12-31")]

# Clean column names if needed (flatten MultiIndex)
sp500_data.columns = [col[0] if isinstance(col, tuple) else col for col in sp500_data.columns]
sp500_data.reset_index(drop=False, inplace=True)
sp500_data = sp500_data.sort_values(by='date', ascending=True)

# Calculate lagged market returns
sp500_data['sp500_Monthly_return'] = sp500_data['4. close'].pct_change(25).shift(1)
sp500_data['sp500_Weekly_return'] = sp500_data['4. close'].pct_change(5).shift(1)
sp500_data['sp500_Daily_return'] = sp500_data['4. close'].pct_change(1).shift(1)

# Retain only final columns
sp500_data = sp500_data[['date', 'sp500_Monthly_return', 'sp500_Weekly_return', 'sp500_Daily_return']]

# Merge S&P 500 returns into main dataset using Offer_date
merged_df = merged_df.merge(
    sp500_data,
    how='left',
    left_on='Offer_date',
    right_on='date'
)
merged_df.drop(columns='date', inplace=True)


In [22]:
# ===========================================
# 12. Fetch and Integrate VIX Index from FRED
# ===========================================
import pandas_datareader.data as web

# Retrieve daily VIX data from FRED
vix = web.DataReader('VIXCLS', 'fred', start='2001-01-01', end='2024-12-31')

# Calculate lagged daily percent change of VIX
vix['VIX_pct_change'] = vix['VIXCLS'].pct_change().shift(1)

# Reset index for merge
vix.reset_index(inplace=True)
vix.rename(columns={'DATE': 'date'}, inplace=True)

# Final columns to retain
vix = vix[['date', 'VIXCLS', 'VIX_pct_change']]

# Merge VIX data using Offer_date
merged_df = merged_df.merge(
    vix,
    how='left',
    left_on='Offer_date',
    right_on='date'
)
merged_df.drop(columns='date', inplace=True)


In [23]:
# ===========================================
# 13. Final Cleanup and Export
# ===========================================

# Drop unused helper columns
merged_df = merged_df.drop(columns=['Merge Date', 'YearMonth', 'Year', 'Month'], errors='ignore')

# Export full enriched dataset
merged_df.to_csv(r"data\Updated SEO ISSUES Dataset\seo_with_factors_Consumption_GDP.csv", index=False)
merged_df.to_parquet(r"data\Updated SEO ISSUES Dataset\seo_with_factors_Consumption_GDP.parquet")

# Export a sample for quick validation and testing
sample_df = merged_df.sample(n=50, random_state=42)
sample_df.to_csv(r"data\Updated SEO ISSUES Dataset\Sample_data_for_validation.csv", index=False)
